# Q# + Jupyter with the QDK

This notebook uses the modern **`qdk`** Python package (successor to the
deprecated `qsharp` package). Importing any `qdk` module registers the
**`%%qsharp`** cell magic, so you can write Q# directly in cells and drive it
from Python.

**Setup (from the repo root, no system-Python installs):**

```bash
mise run setup          # provisions python+uv and runs `uv sync`
mise run lab            # launches Jupyter Lab (uv run jupyter lab)
```

Then open this notebook and select the project's `.venv` kernel.


In [ ]:
# qdk: init the simulator and register the %%qsharp magic
import qdk
from qdk import qsharp
qdk.init()


## 1. Define a Q# operation in a `%%qsharp` cell
The cell magic compiles the Q# and exposes callables under `qdk.code`.


In [ ]:
%%qsharp
import Std.Measurement.*;

operation FlipCoin() : Result {
    use q = Qubit();
    H(q);
    return MResetZ(q);
}


## 2. Run it from Python and tally many shots


In [ ]:
from collections import Counter
results = qsharp.run("FlipCoin()", shots=1000)
counts = Counter(str(r) for r in results)
counts


## 3. Plot the distribution


In [ ]:
import matplotlib.pyplot as plt
plt.bar(list(counts.keys()), list(counts.values()))
plt.title('FlipCoin() — 1000 shots')
plt.ylabel('count')
plt.show()


## 4. Inspect the quantum state directly
`DumpMachine()` prints the simulated state vector — great for debugging.


In [ ]:
%%qsharp
import Std.Diagnostics.*;
import Std.Measurement.*;

operation BellState() : (Result, Result) {
    use (a, b) = (Qubit(), Qubit());
    H(a);
    CNOT(a, b);
    DumpMachine();          // amplitude only on |00> and |11>
    return (MResetZ(a), MResetZ(b));
}


In [ ]:
# The two measurements are always equal — entanglement.
qsharp.run("BellState()", shots=5)


## 5. Sweep a parameter from Python
Classical Python driving a parametrised quantum kernel — the everyday pattern.


In [ ]:
%%qsharp
import Std.Measurement.*;

operation RotateAndMeasure(theta : Double) : Result {
    use q = Qubit();
    Ry(theta, q);
    return MResetZ(q);
}


In [ ]:
import math
for theta in [0.0, math.pi/4, math.pi/2, 3*math.pi/4, math.pi]:
    shots = qsharp.run(f'RotateAndMeasure({theta})', shots=500)
    p_one = sum(1 for r in shots if str(r) == 'One') / len(shots)
    print(f'theta={theta:.3f}  P(One)={p_one:.2f}')


## Next steps

- Work through the runnable files in `../examples/` (phases 2–5 of the ramp).
- See `../CURRICULUM.md` and open `../roadmap.html` to track progress.
- Phase 7: try resource estimation — `from qdk.qre import estimate` — for the
  physical-qubit cost of an algorithm on a fault-tolerant machine.
